# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import os, subprocess

REPO_URL = "https://github.com/Shams-Sajid-Rahman/Sajid_FlyRank_AI"
REPO_DIR = "/content/Sajid_FlyRank_AI"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

Working directory: /content/Sajid_FlyRank_AI


In [2]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"

print("Connected.")

Connected.


In [3]:
raw = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id, f.report_date,
           f.gsc_impressions, f.gsc_clicks, f.gsc_avg_position,
           d.content_updated_date
    FROM {fact_daily} f
    JOIN {dim_content} d USING (content_hash_id)
    WHERE f.report_date >= '2026-02-01' AND f.report_date < '2026-04-01'
""").df()

raw["report_date"] = pd.to_datetime(raw["report_date"])
end_d = raw["report_date"].max()

raw["is_last30"] = raw["report_date"] > (end_d - pd.Timedelta(days=30))
raw["is_prev30"] = ~raw["is_last30"]

print(raw.shape)
print("Date range:", raw["report_date"].min(), "to", raw["report_date"].max())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(17196486, 9)
Date range: 2026-02-01 00:00:00 to 2026-03-31 00:00:00


In [4]:
agg = raw.groupby(["client_hash_id", "content_hash_id"]).apply(
    lambda g: pd.Series({
        "imp_prev30": g.loc[g["is_prev30"], "gsc_impressions"].sum(),
        "imp_last30": g.loc[g["is_last30"], "gsc_impressions"].sum(),
        "clk_prev30": g.loc[g["is_prev30"], "gsc_clicks"].sum(),
        "pos_prev30": g.loc[g["is_prev30"], "gsc_avg_position"].mean(),
        "content_updated_date": g["content_updated_date"].iloc[0],
    })
).reset_index()

# Need enough prior history to define a meaningful trend
agg = agg[agg["imp_prev30"] >= 100].copy()

# CTR from the prior window only (fully known before the outcome window)
agg["ctr_prev30"] = agg["clk_prev30"] / agg["imp_prev30"]

# Staleness at the START of the last30 window (fully known before outcome)
cutoff_date = end_d - pd.Timedelta(days=30)
agg["content_updated_date"] = pd.to_datetime(agg["content_updated_date"])
agg["days_since_update"] = (cutoff_date - agg["content_updated_date"]).dt.days
agg["days_since_update"] = agg["days_since_update"].clip(lower=0)

# LABEL: declining if last30 impressions < 80% of prev30 (same rule as w03)
agg["is_declining_label"] = (agg["imp_last30"] < 0.8 * agg["imp_prev30"]).astype(int)

print(f"{len(agg):,} content items with enough history")
print("Decline rate:", agg["is_declining_label"].mean().round(3))
agg.head()

/tmp/ipykernel_4030/2839860697.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg = raw.groupby(["client_hash_id", "content_hash_id"]).apply(


81,383 content items with enough history
Decline rate: 0.252


,client_hash_id,content_hash_id,imp_prev30,imp_last30,clk_prev30,pos_prev30,content_updated_date,ctr_prev30,days_since_update,is_declining_label
7,client_0797ff3a1fc9a6a5,content_04c67f3541177192,252,325,1,18.820788,2026-02-25,0.003968,4,0
8,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,144,26,0,9.717545,2026-02-25,0.000000,4,1
14,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,125,141,0,8.986303,2026-02-25,0.000000,4,0
18,client_0797ff3a1fc9a6a5,content_1207efddce873942,189,446,0,13.785365,2026-05-20,0.000000,0,0
22,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,167,229,0,11.424053,2026-02-25,0.000000,4,0


Method choice: I use Logistic Regression as an interpretable baseline model
and Random Forest as the stronger candidate, both from this week's toolkit.
This fits a binary decline-prediction task on tabular numeric features
(prior impressions, position, CTR, staleness) — exactly the setting these
methods are built for. Logistic Regression gives clean, interpretable
coefficients; Random Forest can capture non-linear interactions between
signals (e.g. position mattering differently at different impression
levels) without heavy feature engineering. I avoid clustering here since
the task is a labeled prediction problem, not an unsupervised grouping one.

Features used (all knowable before the outcome window starts):
- imp_prev30 (prior 30-day impressions)
- pos_prev30 (prior 30-day average position)
- ctr_prev30 (prior 30-day click-through rate)
- days_since_update (staleness at the start of the outcome window)

Label: is_declining_label (last30 impressions < 80% of prev30) — the
same construction validated in my w03 leakage check.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [5]:
from sklearn.model_selection import GroupShuffleSplit

# Grouped split by client - no client appears in both train and test,
# preventing client-specific patterns from leaking across the split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(agg, groups=agg["client_hash_id"]))

train_df = agg.iloc[train_idx].copy()
test_df = agg.iloc[test_idx].copy()

print(f"Train: {len(train_df):,} rows, {train_df['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test_df):,} rows, {test_df['client_hash_id'].nunique()} clients")

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print(f"Client overlap between train/test: {len(overlap)} (should be 0)")

Train: 55,406 rows, 27 clients
Test:  25,977 rows, 10 clients
Client overlap between train/test: 0 (should be 0)


I use a grouped split by client_hash_id (GroupShuffleSplit, 25% test),
not a random row-level split. Multiple content items belong to the same
client, and clients likely share site-wide patterns (template, niche,
baseline traffic level). A random split could let the model see other
pages from the same client in training and effectively memorize
client-specific quirks rather than learning generalizable signal
patterns. Grouping by client ensures the test set evaluates performance
on clients the model has never seen, which matches the real deployment
scenario: scoring content for new or different clients.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler

feature_cols = ["imp_prev30", "pos_prev30", "ctr_prev30", "days_since_update"]

X_train, y_train = train_df[feature_cols], train_df["is_declining_label"]
X_test, y_test = test_df[feature_cols], test_df["is_declining_label"]

# --- Baseline: your Week-4 rule, applied to the same test set ---
# Recreate the w04 rule score on test_df: normalized (poor position + low visibility)
pos_min, pos_max = train_df["pos_prev30"].min(), train_df["pos_prev30"].max()
imp_min, imp_max = train_df["imp_prev30"].min(), train_df["imp_prev30"].max()

test_df["position_badness"] = (test_df["pos_prev30"] - pos_min) / (pos_max - pos_min)
test_df["low_visibility"] = 1 - (test_df["imp_prev30"] - imp_min) / (imp_max - imp_min)
test_df["baseline_score"] = (test_df["position_badness"] + test_df["low_visibility"]) / 2

baseline_auc = roc_auc_score(y_test, test_df["baseline_score"])

# --- Logistic Regression ---
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_s, y_train)
logreg_auc = roc_auc_score(y_test, logreg.predict_proba(X_test_s)[:, 1])

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

# --- Comparison table ---
results = pd.DataFrame({
    "Model": ["Week-4 baseline rule", "Logistic Regression", "Random Forest"],
    "AUC": [baseline_auc, logreg_auc, rf_auc],
})
print(results.to_string(index=False))

               Model      AUC
Week-4 baseline rule 0.511794
 Logistic Regression 0.596407
       Random Forest 0.612537


Both models beat the Week-4 baseline rule on the same held-out test set
(client-grouped, unseen clients). The baseline rule (0.512 AUC) performs
close to random — it was hand-built on position and impressions without
being validated against an actual future-decline label, so this result
confirms it wasn't doing much real predictive work. Logistic Regression
(0.596) already captures most of the achievable signal from these four
features. Random Forest (0.613) adds a modest further gain, likely from
capturing non-linear interactions (e.g. position mattering more at low
impression levels). The small gap between LogReg and RF suggests the
features themselves, not model complexity, are doing most of the work consistent with a modest but real, non-random relationship between prior
performance signals and future decline.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc")
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

print("Permutation importance (Random Forest):")
print(importance_df.to_string(index=False))
print()

# Look at false positives / false negatives
test_df["rf_pred_proba"] = rf.predict_proba(X_test)[:, 1]
test_df["rf_pred"] = (test_df["rf_pred_proba"] >= 0.5).astype(int)

fp = test_df[(test_df["is_declining_label"] == 0) & (test_df["rf_pred"] == 1)]
fn = test_df[(test_df["is_declining_label"] == 1) & (test_df["rf_pred"] == 0)]

print(f"False positives: {len(fp)} (predicted declining, actually stable)")
print(f"False negatives: {len(fn)} (predicted stable, actually declining)")
print()
print("False positive example stats:")
print(fp[feature_cols].describe().loc[["mean"]])
print()
print("False negative example stats:")
print(fn[feature_cols].describe().loc[["mean"]])

Permutation importance (Random Forest):
          feature  importance
       ctr_prev30    0.075966
       pos_prev30    0.040993
days_since_update    0.011566
       imp_prev30    0.002124

False positives: 75 (predicted declining, actually stable)
False negatives: 6233 (predicted stable, actually declining)

False positive example stats:
      imp_prev30  pos_prev30  ctr_prev30  days_since_update
mean  940.133333   26.069197    0.001011                9.2

False negative example stats:
       imp_prev30  pos_prev30  ctr_prev30  days_since_update
mean  1916.927804    6.962333    0.001765           0.472485


Feature importance (permutation, Random Forest): ctr_prev30 dominates
(0.076), followed by pos_prev30 (0.041). days_since_update and imp_prev30
contribute very little. This means click-through rate not staleness, carries most of the real predictive signal for future decline, which
also explains why my Week-4 rule (built on position and impressions,
without CTR) performed close to random.

Error pattern: the model is heavily skewed toward false negatives
(6,233) versus false positives (75) at a 0.5 threshold, it rarely
predicts decline, so it misses most real declines. Notably, false
negatives have BETTER average position (7.0 vs 26.1) and HIGHER
impressions (1,917 vs 940) than false positives. This means the model's
biggest blind spot is well-performing content that still declines strong current visibility doesn't protect against a coming drop, and
the model currently under-weights that risk. A lower decision threshold,
or optimizing directly for recall/precision-at-K rather than a 0.5 cutoff,
would likely be needed for a real reviewer-facing tool, since missing
6,233 real declines is costlier than 75 false alarms in this use case.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.